In [1]:
import os
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

## Read in all extra data files, Resample to 2 min and combine into one dataframe

In [ ]:
# Could not place raw extra data in git repo due to size limit, so did not push them to Github
extra_path = './networkData/' 

df = pd.DataFrame()

for file in os.listdir(extra_path):
    print(extra_path+file)
    tmp = pd.read_csv(extra_path+file)                                            # Read each CSV file
    tmp['time'] = pd.to_datetime(tmp['time'])                                     # Ensure 'time' is datetime    
    
    numeric_cols = tmp.select_dtypes(include='number').columns                      # Select numeric columns for aggregation
    tmp = tmp.set_index('time').resample('2T')[numeric_cols].mean().reset_index()   # Resample to 2-minute intervals and compute mean
    tmp['District'] = file[0:7]
    
    df = pd.concat([df, tmp])

print(df.shape)
display(df.head())

./networkData/RIY0443_DataTransmission20250813.csv
./networkData/RIY2110_DataTransmission20250813.csv
./networkData/RIY2227_DataTransmission20250813.csv
(10800, 13)


,time,down,up,rnti_count,mcs_down,mcs_down_var,mcs_up,mcs_up_var,rb_down,rb_down_var,rb_up,rb_up_var,District
0,2025-08-13 00:00:00,1.759967e+08,3.632767e+06,5834.000000,12.789461,0.191146,13.991503,0.063371,0.043336,0.000641,0.002007,0.000037,RIY0443
1,2025-08-13 00:02:00,1.959090e+08,1.872591e+06,6721.450000,12.093266,0.104834,13.607168,0.005719,0.062478,0.001129,0.001881,0.000016,RIY0443
2,2025-08-13 00:04:00,2.577617e+08,1.296714e+06,6998.358333,12.228940,0.057763,13.900957,0.007255,0.116035,0.003181,0.000660,0.000016,RIY0443
3,2025-08-13 00:06:00,2.870513e+08,1.188014e+06,7093.191667,13.645386,0.061489,14.075757,0.007397,0.108558,0.004000,0.000654,0.000002,RIY0443
4,2025-08-13 00:08:00,1.567354e+08,3.773311e+06,6948.041667,14.161764,0.057480,13.410258,0.029123,0.091173,0.002269,0.000946,0.000005,RIY0443


In [3]:
df['District'].unique()

array(['RIY0443', 'RIY2110', 'RIY2227'], dtype=object)

`BaseStationID` to `BaseStationName` mapping:

| BaseStationID | BaseStationName |
| ------------- | --------------- |
| RIY0443       | Sulay           |
| RIY2110       | Granada         |
| RIY2227       | Olaya           |


## Load base paper data and combine with extra data

In [5]:
base_path = './dataset/'

df_base = pd.read_csv(base_path+'full_dataset_base.csv')
print(df_base.shape)
display(df_base.head())

(27011, 13)


,time,down,up,rnti_count,mcs_down,mcs_down_var,mcs_up,mcs_up_var,rb_down,rb_down_var,rb_up,rb_up_var,District
0,2018-03-28 15:56:00,174876888.0,1856888.0,10229,15.332298,87.157688,14.981497,49.989484,0.029681,4.497698e-08,0.000541,3.143297e-08,ElBorn
1,2018-03-28 15:58:00,209054184.0,2866200.0,12223,15.116846,87.192168,16.432612,62.494670,0.035971,4.615535e-08,0.000852,4.439640e-08,ElBorn
2,2018-03-28 16:00:00,191464640.0,1935360.0,11152,15.215739,87.227955,15.885238,63.087007,0.032750,4.646104e-08,0.000607,2.993595e-08,ElBorn
3,2018-03-28 16:02:00,241515688.0,2991152.0,14040,15.135400,86.199501,15.714660,77.187459,0.041372,4.532153e-08,0.000925,5.382563e-08,ElBorn
4,2018-03-28 16:04:00,264131088.0,3288816.0,15247,15.188944,86.151119,15.414080,69.118561,0.045074,4.655543e-08,0.001021,5.922178e-08,ElBorn


In [6]:
joined = pd.concat([df_base, df])
print(joined.shape)
print(joined['District'].unique())
display(joined.head())

(37811, 13)
['ElBorn' 'LesCorts' 'PobleSec' 'RIY0443' 'RIY2110' 'RIY2227']


,time,down,up,rnti_count,mcs_down,mcs_down_var,mcs_up,mcs_up_var,rb_down,rb_down_var,rb_up,rb_up_var,District
0,2018-03-28 15:56:00,174876888.0,1856888.0,10229.0,15.332298,87.157688,14.981497,49.989484,0.029681,4.497698e-08,0.000541,3.143297e-08,ElBorn
1,2018-03-28 15:58:00,209054184.0,2866200.0,12223.0,15.116846,87.192168,16.432612,62.494670,0.035971,4.615535e-08,0.000852,4.439640e-08,ElBorn
2,2018-03-28 16:00:00,191464640.0,1935360.0,11152.0,15.215739,87.227955,15.885238,63.087007,0.032750,4.646104e-08,0.000607,2.993595e-08,ElBorn
3,2018-03-28 16:02:00,241515688.0,2991152.0,14040.0,15.135400,86.199501,15.714660,77.187459,0.041372,4.532153e-08,0.000925,5.382563e-08,ElBorn
4,2018-03-28 16:04:00,264131088.0,3288816.0,15247.0,15.188944,86.151119,15.414080,69.118561,0.045074,4.655543e-08,0.001021,5.922178e-08,ElBorn


## Export combined data

In [8]:
joined.to_csv(base_path+'full_dataset_with_extra_data_26Aug.csv', index=False)